In [1]:
import sys
import os
import pandas as pd

In [2]:
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.utils.config_loader import load_config
from src.agent.nodes.critic import critic_node

cfg = load_config()

real_data_path = os.path.join(project_root, cfg.path.data.test)
df = pd.read_csv(real_data_path)
row = df.iloc[501]

In [4]:
print(row) # Dummy data 전체 형태

Unnamed: 0                                                     501
id                                          generation-for-nlp-711
paragraph        현재 가처분 소득이 10,000달러이고 소비 지출이 8,000달러라고 가정하겠습니다...
problems         {'question': '이 정보가 주어지면?', 'choices': ['한계 소비...
question_plus                                                  NaN
Name: 501, dtype: object


In [5]:
import ast

paragraph = row.get('paragraph', '') # 지문

problem = row.get("problems", {})
if isinstance(problem, str):
    problem_dict = ast.literal_eval(problem)
else:
    problem_dict = problem

question = problem_dict.get('question', '') # 질문
choices = problem_dict.get('choices', []) # 선지

print(f"🔹 지문: {paragraph}")
print(f"🔹 질문: {question}")
print(f"🔹 선택지: {choices}")


🔹 지문: 현재 가처분 소득이 10,000달러이고 소비 지출이 8,000달러라고 가정하겠습니다. 가처분 소득이 100달러 증가할 때마다 저축액은 10달러 증가합니다.
🔹 질문: 이 정보가 주어지면?
🔹 선택지: ['한계 소비 성향은 0.80이다.', '한계 저축 성향은 0.20이다.', '한계 저축 성향은 0.10이다.', '한계 저축 성향은 0.90이다.']


In [6]:
dummy_state = {
    "paragraph": paragraph,
    "problem": {
        "question": question,
        "choices": choices
    },
    "solver_results": [
        {
            # 케이스 1: 정답(3번: 0.10)은 맞지만 근거가 빈약함 (논리 부족)
            "answer": "3",
            "reasoning": "지문에서 소득과 저축의 변화가 언급되어 있으며, 보기 중 0.10이 가장 수치적으로 타당해 보입니다." 
        },
        {
            # 케이스 2: 정답이 틀렸으며(1번) 근거도 오류인 경우 (계산 실수/논리 오류)
            "answer": "1",
            "reasoning": "현재 가처분 소득 10,000달러 중 소비 지출이 8,000달러이므로 비율은 0.8입니다. 따라서 소득 증가 시 소비 성향도 0.80이 됩니다." 
        },
        {
            # 케이스 3: 정답(3번)과 근거(MPS 계산)가 모두 완벽함 (Pass 기대)
            "answer": "3",
            "reasoning": "한계 저축 성향(MPS)은 소득 변화량에 대한 저축 변화량의 비율입니다. 지문에 따르면 소득 100달러 증가 시 저축이 10달러 증가하므로, MPS = 10 / 100 = 0.10입니다. 따라서 '한계 저축 성향은 0.10이다'라는 3번 선지가 정답입니다." 
        }
    ]
}

In [7]:
# 노드 초기화
print("⏳ CriticNode 초기화 중... (모델 로딩)")
CriticNode = critic_node(cfg)
# 노드 실행
print(f"▶️ 검증 시작 (총 {len(dummy_state['solver_results'])}개의 결과)")
result = CriticNode(dummy_state)

⏳ CriticNode 초기화 중... (모델 로딩)
🔧 [critic_node] 초기화 중... (사용 모델: main_solver)
🔄 [Loader] 모델 로딩 시작: Qwen3-32B-4bit (unsloth/Qwen3-32B-bnb-4bit)
   ↳ ⚡ 양자화 설정 적용 중...


/data/ephemeral/home/dh_dir/pro-nlp-generationfornlp-nlp-08/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ [Loader] 로딩 완료!
▶️ 검증 시작 (총 3개의 결과)


In [8]:
# 결과 출력
print("\n" + "="*50)
print("🧐 CriticNode 검증 결과 요약")
print("="*50)

for i, report in enumerate(result['critic_results']):
    orig_res = dummy_state['solver_results'][i]
    print(f"\n[결과 {i+1}]")
    print(f"🔹 모델 답변: {orig_res['answer']}")
    print(f"🔹 선정 이유: {orig_res['reasoning']}")
    print(f"--------------------------------------------------")
    print(f"🚩 판정: {report['critic_result']}")
    print(f"📝 근거: {report['critic_reason']}")
    print("-" * 50)


🧐 CriticNode 검증 결과 요약

[결과 1]
🔹 모델 답변: 3
🔹 선정 이유: 지문에서 소득과 저축의 변화가 언급되어 있으며, 보기 중 0.10이 가장 수치적으로 타당해 보입니다.
--------------------------------------------------
🚩 판정: Pass
📝 근거: 지문에 따르면 가처분 소득 100달러 증가 시 저축액 10달러 증가하므로 한계 저축 성향(MPS)은 10/100=0.10이다. 모델의 정답 선정 이유가 수치 계산에 기반한 논리적 근거를 제시하였다.
--------------------------------------------------

[결과 2]
🔹 모델 답변: 1
🔹 선정 이유: 현재 가처분 소득 10,000달러 중 소비 지출이 8,000달러이므로 비율은 0.8입니다. 따라서 소득 증가 시 소비 성향도 0.80이 됩니다.
--------------------------------------------------
🚩 판정: Fail
📝 근거: 모델은 초기 소비 비율(8,000/10,000)을 기준으로 한계 소비 성향을 산출했으나, 문제에서 주어진 핵심 정보는 '소득 증가 시 저축 증가율(100달러당 10달러)'입니다. 이 정보를 기반으로 한계 저축 성향(MPS)은 0.10, 한계 소비 성향(MPC)은 1-0.10=0.90이 되어야 합니다. 모델은 지문의 핵심 조건을 무시하고 정답을 도출했습니다.
--------------------------------------------------

[결과 3]
🔹 모델 답변: 3
🔹 선정 이유: 한계 저축 성향(MPS)은 소득 변화량에 대한 저축 변화량의 비율입니다. 지문에 따르면 소득 100달러 증가 시 저축이 10달러 증가하므로, MPS = 10 / 100 = 0.10입니다. 따라서 '한계 저축 성향은 0.10이다'라는 3번 선지가 정답입니다.
--------------------------------------------------
🚩 판정: Pass